# Module symptômes

Le module transforme une description libre en français en une liste de symptômes de la base de connaissances.
Méthode : racinisation (Snowball), correspondance approximative (RapidFuzz) pour les fautes de frappe,
et détection des négations (« ne vibre pas », « aucune fuite »). Code : `diagnostic/symptomes.py`.

Cliquez sur **Run All**.

In [ ]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import pandas as pd

from diagnostic.symptomes import CORPUS_DEV, CORPUS_TEST, ExtracteurSymptomes, evaluer

pd.set_option("display.max_colwidth", None)
extracteur = ExtracteurSymptomes()

## 1. Exemples

Pour chaque phrase : les symptômes reconnus, l'expression qui les a déclenchés, et les symptômes niés.

In [ ]:
EXEMPLES = [
    "La pompe vibre énormément et sent le brûlé.",
    "Les roulements grincent quand la vitesse augmente.",
    "Ça ne vibre pas mais ça grince.",
    "Aucune fuite, par contre le moteur chauffe.",
    "La machine vibrre.",  # faute de frappe
]
lignes = []
for texte in EXEMPLES:
    r = extracteur.extraire(texte)
    lignes.append({
        "Description": texte,
        "Symptômes reconnus": ", ".join(f"{s['libelle']} (« {s['expression']} », {s['confiance']:.2f})" for s in r["presents"]) or "—",
        "Symptômes niés": ", ".join(s["libelle"] for s in r["absents"]) or "—",
    })
pd.DataFrame(lignes).style.hide(axis="index")

## 2. Évaluation sur les corpus annotés

- **Développement** : phrases utilisées pour mettre au point le module.
- **Test** : phrases rédigées à part, pour l'évaluation finale.

Critère visé : au moins 85 % des symptômes annotés sont extraits (rappel).

In [ ]:
resultats = {}
for nom, chemin in [("Développement", CORPUS_DEV), ("Test", CORPUS_TEST)]:
    if chemin.exists():
        resultats[nom] = evaluer(chemin, extracteur)

pd.DataFrame({nom: {"Phrases": r["phrases"], "Phrases entièrement correctes": r["parfaites"],
                    "Rappel": f"{r['rappel']:.1%}", "Précision": f"{r['precision']:.1%}", "F1": f"{r['f1']:.1%}",
                    "Critère (rappel ≥ 85 %)": "ATTEINT" if r["rappel"] >= 0.85 else "NON ATTEINT"}
              for nom, r in resultats.items()})

## 3. Erreurs

In [ ]:
erreurs = [{"Corpus": nom, "Phrase": texte, "Manqués": ", ".join(manques) or "—", "En trop": ", ".join(en_trop) or "—"}
           for nom, r in resultats.items() for texte, manques, en_trop in r["erreurs"]]
pd.DataFrame(erreurs).style.hide(axis="index") if erreurs else print("Aucune erreur")